# Sanitization refers to the following operation:
> performed on the ./translation/data/method1/clean/final/

### 1. The word should be identified as a single token by the Sarvam Tokeniser.

In [51]:
from transformers import AutoTokenizer
models_to_test = {
    "Sarvam-2B": "sarvamai/sarvam-2b-v0.5",
    "Qwen-2.5-1.5B": "Qwen/Qwen2.5-1.5B",
    "Gemma-2-2B": "google/gemma-2-2b",
    "Llama-2-7B": "meta-llama/Llama-2-7b-hf"
}

In [62]:
import pandas as pd
import numpy as np

In [63]:
prefix = "translation/data/method1/clean/final/"
files = {
    'ben': f'{prefix}ben.csv',
    'mar': f'{prefix}mar.csv',
    'tel': f'{prefix}tel.csv',
    'tam': f'{prefix}tam.csv'
}
word_columns = {
    'en' : None,
    'hi' : None,
    'ben': None,
    'mar': None,
    'tel': None,
    'tam': None
}

df = pd.read_csv(files["ben"])
word_columns['en'] = df["en_word"]
word_columns['hi'] = df["hi_word"]

for lang_code, fname in files.items():
    df = pd.read_csv(fname)
    word_columns[lang_code] = df[f"{lang_code}_word"]

In [64]:
```
def analyze_multi_token_words(label, model_id):
    results_summary = {}
    print(f"\nEvaluating tokenizer for {label} ({model_id})...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    multi_token_words = {lang: [] for lang in word_columns}
    
    for lang_code, words in word_columns.items():
        for word in words:
            # Skip NaNs or empty strings if any exist
            if not isinstance(word, str):
                continue
            
            ids = tokenizer.encode(word, add_special_tokens=False)
            if len(ids) != 1:
                multi_token_words[lang_code].append((word, len(ids)))
                
        print(f"  [{lang_code}] Multi-token words: {len(multi_token_words[lang_code])} / {len(words)}")
        
    results_summary[label] = {lang: len(tokens) for lang, tokens in multi_token_words.items()}
```

SyntaxError: invalid syntax (1099781863.py, line 1)

In [65]:
def analyze_multi_token_words(label, model_id, word_columns):
    print(f"\nEvaluating tokenizer: {label} ({model_id})")
    print("-" * 65)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    model_stats = {}
    
    for lang_code, words in word_columns.items():
        all_token_counts = []
        multi_token_words = []
        
        for word in words:
            if not isinstance(word, str) or not word.strip():
                continue
            
            ids = tokenizer.encode(word, add_special_tokens=False)
            count = len(ids)
            all_token_counts.append(count)
            
            if count > 1:
                multi_token_words.append((word, count))
                
        total_valid = len(all_token_counts)
        multi_count = len(multi_token_words)
        
        # 1. Overall average (Fertility across full lexicon)
        avg_all = float(np.mean(all_token_counts)) if total_valid > 0 else 0.0
        
        # 2. Multi-token only average (Fragmentation severity)
        multi_lengths = [count for _, count in multi_token_words]
        avg_multi_only = float(np.mean(multi_lengths)) if multi_count > 0 else 1.0
        
        model_stats[lang_code] = {
            "total_words": total_valid,
            "multi_token_count": multi_count,
            "pct_multi_token": (multi_count / total_valid * 100) if total_valid > 0 else 0.0,
            "avg_tokens_all": round(avg_all, 3),
            "avg_tokens_multi_only": round(avg_multi_only, 3)
        }
        
        print(f"[{lang_code:>3}] Total: {total_valid:<4} | "
              f"Multi-token: {multi_count:<3} ({model_stats[lang_code]['pct_multi_token']:5.1f}%) | "
              f"Avg (All): {avg_all:.2f} | "
              f"Avg (Multi-only): {avg_multi_only:.2f}")
              
    return model_stats

In [68]:
all_results = {}
for label, model_id in models_to_test.items():
    all_results[label] = analyze_multi_token_words(label, model_id, word_columns)


Evaluating tokenizer: Sarvam-2B (sarvamai/sarvam-2b-v0.5)
-----------------------------------------------------------------
[ en] Total: 132  | Multi-token: 4   (  3.0%) | Avg (All): 1.03 | Avg (Multi-only): 2.00
[ hi] Total: 132  | Multi-token: 15  ( 11.4%) | Avg (All): 1.17 | Avg (Multi-only): 2.47
[ben] Total: 132  | Multi-token: 39  ( 29.5%) | Avg (All): 1.38 | Avg (Multi-only): 2.28
[mar] Total: 132  | Multi-token: 34  ( 25.8%) | Avg (All): 1.34 | Avg (Multi-only): 2.32
[tel] Total: 132  | Multi-token: 51  ( 38.6%) | Avg (All): 1.54 | Avg (Multi-only): 2.39
[tam] Total: 132  | Multi-token: 62  ( 47.0%) | Avg (All): 1.53 | Avg (Multi-only): 2.13

Evaluating tokenizer: Qwen-2.5-1.5B (Qwen/Qwen2.5-1.5B)
-----------------------------------------------------------------
[ en] Total: 132  | Multi-token: 16  ( 12.1%) | Avg (All): 1.12 | Avg (Multi-only): 2.00
[ hi] Total: 132  | Multi-token: 132 (100.0%) | Avg (All): 4.55 | Avg (Multi-only): 4.55
[ben] Total: 132  | Multi-token: 132 (10

In [69]:
# Reshape into a side-by-side comparison DataFrame for Average Tokens (All words)
df_avg_all = pd.DataFrame({
    model: {lang: stats["avg_tokens_all"] for lang, stats in langs.items()}
    for model, langs in all_results.items()
})

# Reshape into a side-by-side comparison DataFrame for Multi-Token Counts
df_multi_counts = pd.DataFrame({
    model: {lang: stats["multi_token_count"] for lang, stats in langs.items()}
    for model, langs in all_results.items()
})

print("\n=== Average Tokens Per Word (All Words - Lower is Better) ===")
print(df_avg_all)

print("\n=== Total Multi-Token Words Count ===")
print(df_multi_counts)


=== Average Tokens Per Word (All Words - Lower is Better) ===
     Sarvam-2B  Qwen-2.5-1.5B  Gemma-2-2B  Llama-2-7b
en       1.030          1.121       1.015       1.008
hi       1.167          4.553       2.470       5.924
ben      1.379          5.083       3.250       7.318
mar      1.341          4.538       2.720       6.000
tel      1.538          8.985       3.720      15.992
tam      1.530          6.167       3.144       7.462

=== Total Multi-Token Words Count ===
     Sarvam-2B  Qwen-2.5-1.5B  Gemma-2-2B  Llama-2-7b
en           4             16           2           1
hi          15            132         126         132
ben         39            132         131         132
mar         34            132         128         132
tel         51            132         132         132
tam         62            132         131         132
